# 48 — Fail-closed production gallery

Exercise concrete rejection paths in SC-NeuroCore production APIs. Every bad
input below must raise; the notebook contains no mirrored validator and no
fallback implementation.

## Honesty box

| | |
|---|---|
| **Proves** | `SCNIRConversionConfig` and `StochasticSTDPSynapse` reject the demonstrated malformed inputs through production code. |
| **Does not prove** | Exhaustive security coverage, formal correctness, or guard behaviour in unrelated APIs. |
| **Components** | SC-NIR conversion configuration and stochastic STDP synapse. |


In [ ]:
from __future__ import annotations

from collections.abc import Callable

from sc_neurocore.ir import SCNIRConversionConfig
from sc_neurocore.synapses import StochasticSTDPSynapse

print("SC-NeuroCore — NB-48 fail-closed production gallery")


In [ ]:
def require_refusal(label: str, operation: Callable[[], object]) -> str:
    try:
        operation()
    except (TypeError, ValueError) as exc:
        detail = f"{type(exc).__name__}: {exc}"
        print(f"REFUSED {label}: {detail}")
        return detail
    raise AssertionError(f"production guard accepted invalid case: {label}")


refusals = {
    "zero_bitstream_length": require_refusal(
        "zero SC-NIR bitstream length",
        lambda: SCNIRConversionConfig(bitstream_length=0),
    ),
    "fraction_not_below_width": require_refusal(
        "SC-NIR fraction equal to data width",
        lambda: SCNIRConversionConfig(bitstream_length=256, data_width=16, fraction=16),
    ),
    "zero_stdp_window": require_refusal(
        "zero STDP window",
        lambda: StochasticSTDPSynapse(
            w_min=0.0, w_max=1.0, w=0.5, length=64, window_size=0
        ),
    ),
}


In [ ]:
valid_config = SCNIRConversionConfig(bitstream_length=256)
valid_synapse = StochasticSTDPSynapse(
    w_min=0.0, w_max=1.0, w=0.5, length=64, window_size=8, seed=7
)
refusals["non_binary_pre_bit"] = require_refusal(
    "non-binary STDP pre bit", lambda: valid_synapse.process_step(2, 0)
)
valid_output = valid_synapse.process_step(1, 1)
assert valid_output in (0, 1)
print(
    f"valid SC-NIR length={valid_config.bitstream_length}; "
    f"valid STDP output={valid_output}; refusals={len(refusals)}"
)
print("NB-48 complete: four production refusal paths, no fallback.")

